<a href="https://colab.research.google.com/github/maivu18082005-hub/Parameter_Identification_of_PMSM_using_PINN/blob/main/03_train_mlp.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ===== CELL 1: KẾT NỐI DRIVE & KIỂM TRA GPU =====
from google.colab import drive  # Import thư viện drive của Colab để gắn kết nối tới Google Drive[cite: 4]
import torch                    # Import thư viện PyTorch làm framework chính cho Deep Learning
import os                       # Import thư viện os để thực hiện các thao tác kiểm tra thư mục và đường dẫn[cite: 4]

drive.mount('/content/drive')   # Thực hiện Mount (gắn) Google Drive của bạn vào môi trường ảo Colab[cite: 4]

# Khai báo đường dẫn thư mục dự án gốc trên Google Drive[cite: 4]
DRIVE_PROJECT = '/content/drive/MyDrive/Parameter_Identification_of_PMSM_using_PINN'

# Khai báo đường dẫn chứa tập dữ liệu đã qua tiền xử lý (.npy và .pkl)[cite: 3]
PROCESSED_DATA_DIR = os.path.join(DRIVE_PROJECT, 'dataset', 'processed')

# Khai báo thư mục lưu trữ trọng số mô hình tốt nhất (Checkpoints)[cite: 3, 4]
CHECKPOINT_DIR = os.path.join(DRIVE_PROJECT, 'checkpoints')

# Tự động tạo thư mục checkpoints trên Google Drive nếu thư mục này chưa tồn tại[cite: 3]
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

# Kiểm tra xem Colab có GPU không: nếu có thì dùng 'cuda', nếu không thì dùng 'cpu'
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print('Thiết bị tính toán đang dùng:', device)                 # In ra màn hình thiết bị đang chạy (ví dụ: cuda:0)
print('Thư mục dữ liệu processed tồn tại:', os.path.exists(PROCESSED_DATA_DIR)) # Kiểm tra thư mục dữ liệu đã sẵn sàng chưa[cite: 3]

# ===== THÊM ĐOẠN NÀY VÀO CELL 1 ĐỂ CỐ ĐỊNH KẾT QUẢ =====
#import random
#import numpy as np
#import torch

#def set_seed(seed=42):
#    random.seed(seed)
#    np.random.seed(seed)
#    torch.manual_seed(seed)
#    torch.cuda.manual_seed(seed)
#    torch.cuda.manual_seed_all(seed)
#    torch.backends.cudnn.deterministic = True
#    torch.backends.cudnn.benchmark = False

#set_seed(42)  # Cố định hạt giống ngẫu nhiên là 42

Mounted at /content/drive
Thiết bị tính toán đang dùng: cuda
Thư mục dữ liệu processed tồn tại: True


In [ ]:
# ===== CELL 2: NẠP TOÀN BỘ DỮ LIỆU TRỰC TIẾP LÊN VRAM GPU (SIÊU TỐC) =====
import numpy as np  # Import NumPy để đọc các file mảng .npy từ Google Drive

# Nạp trực tiếp dữ liệu tập Train lên bộ nhớ GPU (device='cuda')
X_train_t = torch.tensor(np.load(os.path.join(PROCESSED_DATA_DIR, 'X_train.npy')), dtype=torch.float32, device=device)
Y_train_t = torch.tensor(np.load(os.path.join(PROCESSED_DATA_DIR, 'Y_train.npy')), dtype=torch.float32, device=device)

# Nạp trực tiếp dữ liệu tập Validation lên bộ nhớ GPU
X_val_t = torch.tensor(np.load(os.path.join(PROCESSED_DATA_DIR, 'X_validation.npy')), dtype=torch.float32, device=device)
Y_val_t = torch.tensor(np.load(os.path.join(PROCESSED_DATA_DIR, 'Y_validation.npy')), dtype=torch.float32, device=device)

# Nạp trực tiếp dữ liệu tập Test lên bộ nhớ GPU
X_test_t = torch.tensor(np.load(os.path.join(PROCESSED_DATA_DIR, 'X_test.npy')), dtype=torch.float32, device=device)
Y_test_t = torch.tensor(np.load(os.path.join(PROCESSED_DATA_DIR, 'Y_test.npy')), dtype=torch.float32, device=device)

print(f'Đã nạp thẳng lên GPU ({device})! Train: {len(X_train_t)} | Validation: {len(X_val_t)} | Test: {len(X_test_t)}')

Đã nạp thẳng lên GPU (cuda)! Train: 89974 | Validation: 29990 | Test: 29995


In [ ]:
# ===== CELL 3: KIẾN TRÚC MẠNG MLP =====
import torch.nn as nn # Import module chứa các lớp mạng Neural của PyTorch

class PMSM_MLP(nn.Module):
    """
    Kiến trúc Mạng Multi-Layer Perceptron (MLP) thuần dữ liệu (Data-driven).
    Nhận 5 nút đầu vào [ud, uq, Id, Iq, nm] và dự đoán 3 nút đầu ra [Rs, Ld, Lq].[cite: 2, 3]
    """
    def __init__(self, input_dim=5, output_dim=3, hidden_dim=128):
        super(PMSM_MLP, self).__init__()

        # Xây dựng các lớp liên kết đầy đủ (Fully Connected / Dense Layers)
        self.network = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),   # Lớp đầu vào: 5 nút -> 128 nút ẩn
            nn.Tanh(),                          # Hàm kích hoạt Tanh tạo thuộc tính phi tuyến
            nn.Linear(hidden_dim, hidden_dim),  # Lớp ẩn thứ 1: 128 -> 128 nút
            nn.Tanh(),                          # Hàm kích hoạt Tanh
            nn.Linear(hidden_dim, hidden_dim),  # Lớp ẩn thứ 2: 128 -> 128 nút
            nn.Tanh(),                          # Hàm kích hoạt Tanh
            nn.Linear(hidden_dim, output_dim)   # Lớp đầu ra: 128 -> 3 tham số ước lượng [Rs, Ld, Lq][cite: 2, 3]
        )

    def forward(self, x):
        # Hàm lan truyền tiến (Forward Pass) truyền tensor x qua các lớp mạng
        return self.network(x)

# Khởi tạo mô hình MLP và chuyển toàn bộ thông số lên GPU (cuda)
model = PMSM_MLP(input_dim=5, output_dim=3, hidden_dim=128).to(device)
print(model) # In cấu trúc mạng MLP ra màn hình

PMSM_MLP(
  (network): Sequential(
    (0): Linear(in_features=5, out_features=128, bias=True)
    (1): Tanh()
    (2): Linear(in_features=128, out_features=128, bias=True)
    (3): Tanh()
    (4): Linear(in_features=128, out_features=128, bias=True)
    (5): Tanh()
    (6): Linear(in_features=128, out_features=3, bias=True)
  )
)


In [ ]:
# ===== CELL 4: HUẤN LUYỆN MÔ HÌNH MLP TRỰC TIẾP TRÊN GPU =====
import torch.optim as optim  # Import bộ tối ưu hóa trọng số Optimizer
import shutil                # Import thư viện shutil để copy file checkpoint sang Google Drive
import time                  # Import thư viện time để đo tổng thời gian huấn luyện

criterion = nn.MSELoss()                             # Hàm mất mát sai số bình phương trung bình (MSE)
optimizer = optim.Adam(model.parameters(), lr=0.002) # Thuật toán tối ưu Adam (đồng bộ lr = 0.002 với PINN)
# Bộ điều chỉnh tự động giảm một nửa Learning Rate nếu Val Loss không cải thiện sau 15 epochs
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=15)

epochs = 150                 # Tổng số vòng lặp huấn luyện (đồng bộ 150 epochs với PINN)
batch_size = 512             # Kích thước mỗi lô dữ liệu xử lý song song trên GPU
n_train = len(X_train_t)     # Tổng số mẫu trong tập Train
best_val_loss = float('inf') # Khởi tạo kỷ lục Val Loss thấp nhất

# Lưu file checkpoint tạm thời trên ổ cứng cục bộ Colab (/content) để tránh nghẽn mạng Drive
LOCAL_CKPT = '/content/best_mlp_model.pth'
DRIVE_CKPT = os.path.join(CHECKPOINT_DIR, 'best_mlp_model.pth')

print("=== BẮT ĐẦU HUẤN LUYỆN MLP (GPU-NATIVE) ===")
start_time = time.time()  # Bấm giờ bắt đầu huấn luyện

for epoch in range(1, epochs + 1):
    # --- GIAI ĐOẠN 1: HUẤN LUYỆN (TRAIN) ---
    model.train()  # Bật chế độ huấn luyện
    # Tạo hoán vị chỉ số ngẫu nhiên trực tiếp trên GPU để xáo trộn dữ liệu mỗi Epoch
    perm = torch.randperm(n_train, device=device)
    train_loss = 0.0

    # Duyệt qua từng Batch dữ liệu trực tiếp trên VRAM của GPU
    for i in range(0, n_train, batch_size):
        idx = perm[i:i + batch_size]  # Lấy danh sách chỉ số của Batch hiện tại
        inputs = X_train_t[idx]       # Lấy lô dữ liệu đầu vào [ud, uq, Id, Iq, nm]
        targets = Y_train_t[idx]      # Lấy lô nhãn thực tế [Rs, Ld, Lq]

        optimizer.zero_grad()              # Đặt lại gradient về 0
        outputs = model(inputs)            # Lan truyền tiến: dự đoán tham số [Rs, Ld, Lq]
        loss = criterion(outputs, targets) # Tính sai số MSE thuần dữ liệu
        loss.backward()                    # Lan truyền ngược tính gradient
        optimizer.step()                   # Cập nhật trọng số mạng

        train_loss += loss.item() * len(idx) # Cộng dồn sai số để thống kê

    train_loss /= n_train  # Tính sai số MSE trung bình trên toàn tập Train

    # --- GIAI ĐOẠN 2: ĐÁNH GIÁ TRÊN VALIDATION ---
    model.eval()  # Chuyển mô hình sang chế độ đánh giá
    with torch.no_grad():  # Tắt tính toán gradient để tiết kiệm bộ nhớ
        val_outputs = model(X_val_t)                   # Dự đoán toàn bộ tập Validation cùng lúc trên GPU
        val_loss = criterion(val_outputs, Y_val_t).item() # Tính sai số MSE trên tập Validation

    scheduler.step(val_loss)  # Cập nhật bộ điều chỉnh Learning Rate theo Val Loss

    # Lưu trọng số mô hình tốt nhất vào ổ cứng cục bộ /content
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_loss': val_loss
        }, LOCAL_CKPT)

    # In thông số huấn luyện và thời gian thực tế sau mỗi 15 epochs
    if epoch % 15 == 0 or epoch == 1:
        elapsed = time.time() - start_time
        print(f"Epoch [{epoch:3d}/{epochs}] | Train Loss (MSE): {train_loss:.6f} | Val Loss (MSE): {val_loss:.6f} | Thời gian: {elapsed:.1f} giây")

# Copy file trọng số tốt nhất từ Colab sang Google Drive 1 lần duy nhất sau khi hoàn tất
shutil.copy(LOCAL_CKPT, DRIVE_CKPT)
print(f"\n--> Hoàn thành 150 epochs trong {time.time() - start_time:.1f} giây! Đã lưu vào: {DRIVE_CKPT}")

=== BẮT ĐẦU HUẤN LUYỆN MLP (GPU-NATIVE) ===
Epoch [  1/150] | Train Loss (MSE): 0.328024 | Val Loss (MSE): 0.547925 | Thời gian: 1.1 giây
Epoch [ 15/150] | Train Loss (MSE): 0.176648 | Val Loss (MSE): 0.733085 | Thời gian: 5.9 giây
Epoch [ 30/150] | Train Loss (MSE): 0.160192 | Val Loss (MSE): 0.785576 | Thời gian: 10.9 giây
Epoch [ 45/150] | Train Loss (MSE): 0.152071 | Val Loss (MSE): 0.645039 | Thời gian: 16.6 giây
Epoch [ 60/150] | Train Loss (MSE): 0.143241 | Val Loss (MSE): 0.694557 | Thời gian: 21.6 giây
Epoch [ 75/150] | Train Loss (MSE): 0.136784 | Val Loss (MSE): 0.691453 | Thời gian: 26.6 giây
Epoch [ 90/150] | Train Loss (MSE): 0.132499 | Val Loss (MSE): 0.669140 | Thời gian: 32.3 giây
Epoch [105/150] | Train Loss (MSE): 0.129740 | Val Loss (MSE): 0.656442 | Thời gian: 37.6 giây
Epoch [120/150] | Train Loss (MSE): 0.128131 | Val Loss (MSE): 0.661384 | Thời gian: 42.9 giây
Epoch [135/150] | Train Loss (MSE): 0.127199 | Val Loss (MSE): 0.657916 | Thời gian: 48.3 giây
Epoch [1

In [ ]:
# ===== CELL 5: ĐÁNH GIÁ MLP TRÊN TẬP TEST =====
import joblib  # Import joblib để nạp file lưu bộ giải chuẩn hóa Scaler

# 1. Nạp lại bộ Scaler của Target đã được fit ở bước tiền xử lý
Y_scaler = joblib.load(os.path.join(PROCESSED_DATA_DIR, 'Y_scaler.pkl'))

# 2. Nạp bản trọng số mô hình tốt nhất đã lưu trên Google Drive
checkpoint = torch.load(os.path.join(CHECKPOINT_DIR, 'best_mlp_model.pth'))
model.load_state_dict(checkpoint['model_state_dict']) # Khôi phục trọng số tối ưu
model.eval()                                          # Chuyển mô hình sang chế độ dự đoán

# 3. Dự đoán toàn bộ tập Test trực tiếp trên GPU và chuyển kết quả về CPU dạng mảng NumPy
with torch.no_grad():
    all_preds = model(X_test_t).cpu().numpy()
    all_targets = Y_test_t.cpu().numpy()

# 4. Giải chuẩn hóa (Inverse Transform) đưa kết quả về đơn vị vật lý thực (Ohm cho Rs, Henry cho Ld và Lq)
preds_real = Y_scaler.inverse_transform(all_preds)
targets_real = Y_scaler.inverse_transform(all_targets)

# 5. Tính sai số phần trăm tuyệt đối trung bình (MAPE) cho từng tham số Rs, Ld, Lq
mape = np.mean(np.abs((targets_real - preds_real) / targets_real), axis=0) * 100

print("=== KẾT QUẢ ĐÁNH GIÁ MLP TRÊN TẬP TEST ===")
print(f"Sai số Rs trung bình: {mape[0]:.2f}%") # In % sai số của điện trở stator Rs
print(f"Sai số Ld trung bình: {mape[1]:.2f}%") # In % sai số của độ tự cảm trục d Ld
print(f"Sai số Lq trung bình: {mape[2]:.2f}%") # In % sai số của độ tự cảm trục q Lq

=== KẾT QUẢ ĐÁNH GIÁ MLP TRÊN TẬP TEST ===
Sai số Rs trung bình: 6.23%
Sai số Ld trung bình: 0.09%
Sai số Lq trung bình: 0.19%
